In [3]:
!wget https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

--2026-10-05 21:04:08--  https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
Resolving storage.googleapis.com (storage.googleapis.com)... 2404:6800:4013:813::cf, 2404:6800:4013:802::cf, 2404:6800:4013:800::cf, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|2404:6800:4013:813::cf|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7819105 (7.5M) [application/octet-stream]
Saving to: ‘hand_landmarker.task’

hand_landmarker.tas 100%[===================>]   7.46M  2.76MB/s    in 2.7s    

2026-10-05 21:04:10 (2.76 MB/s) - ‘hand_landmarker.task’ saved [7819105/7819105]



In [12]:
import sys

!{sys.executable} -m pip install mediapipe

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3.13 install --upgrade pip


In [15]:
import sys

!{sys.executable} -m pip install opencv-python

Defaulting to user installation because normal site-packages is not writeable
  Using cached opencv_python-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl (73.8 MB)

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3.13 install --upgrade pip


In [2]:
import cv2
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [3]:
import os

print(os.path.exists("hand_landmarker.task"))

True


In [6]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt

from IPython.display import display, clear_output

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [16]:
base_options = python.BaseOptions(
    model_asset_path="hand_landmarker.task"
)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.1,
    min_hand_presence_confidence=0.1,
    min_tracking_confidence=0.1
)

detector = vision.HandLandmarker.create_from_options(options)

W0000 00:00:1791374498.624317   34985 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791374498.802185   34986 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [19]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open webcam")

else:

    try:

        while True:

            # Capture frame
            ret, frame = cap.read()

            if not ret:
                print("Cannot read frame")
                break

            # -----------------------------
            # 3. Convert BGR → RGB
            # -----------------------------

            rgb_frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            # -----------------------------
            # 4. Convert to MediaPipe Image
            # -----------------------------

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb_frame
            )

            # -----------------------------
            # 5. Detect hands
            # -----------------------------

            result = detector.detect(mp_image)

            # -----------------------------
            # 6. Draw landmarks
            # -----------------------------

            if result.hand_landmarks:

                for hand in result.hand_landmarks:

                    for landmark in hand:

                        # MediaPipe coordinates are normalized
                        x = int(
                            landmark.x * frame.shape[1]
                        )

                        y = int(
                            landmark.y * frame.shape[0]
                        )

                        cv2.circle(
                            frame,
                            (x, y),
                            5,
                            (0, 255, 0),
                            -1
                        )

            # -----------------------------
            # 7. Display
            # -----------------------------

            # frame_rgb = cv2.cvtColor(
            #     frame,
            #     cv2.COLOR_BGR2RGB
            # )

            # clear_output(wait=True)

            # plt.figure(figsize=(10, 6))
            # plt.imshow(frame_rgb)
            # plt.axis("off")
            # display(plt.gcf())
            # plt.close()

    except KeyboardInterrupt:

        print("Webcam stopped")

    finally:

        cap.release()
        detector.close()

Webcam stopped
